# 1. Load data from NAB

In [1]:
import os
import json
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

In [2]:
NAB_ROOT = "NAB"
DATA_DIR = os.path.join(NAB_ROOT, 'data', 'realAWSCloudwatch')
WINDOWS_FILE = os.path.join(NAB_ROOT, 'labels', 'combined_windows.json')

DATA_DIR, WINDOWS_FILE

('NAB\\data\\realAWSCloudwatch', 'NAB\\labels\\combined_windows.json')

DB config

In [12]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'metrics_db',
    'user': 'mlops',
    'password': 'mlops123',
}

In [25]:
def load_windows() -> dict:
    """
    Windows file chứa ground truth

    `Key: (str)`: Tên đường dẫn tới file dữ liệu csv (VD: `'realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv'`).

    `Value (list)`: Danh sách các khoảng thời gian bất thường (anomaly windows) của file đó.

    `[]` (List rỗng): File hoàn toàn bình thường, không có bất thường nào (VD: các file trong thư mục artificialNoAnomaly).

    `[['start', 'end'], ...]`: Có bất thường. Mỗi phần tử là một list con chứa chính xác 2 chuỗi thời gian: [Thời gian bắt đầu, Thời gian kết thúc].
    """
    with open(WINDOWS_FILE) as f:
        return json.load(f)

In [5]:
import pprint

pprint.pprint(load_windows())

{'artificialNoAnomaly/art_daily_no_noise.csv': [],
 'artificialNoAnomaly/art_daily_perfect_square_wave.csv': [],
 'artificialNoAnomaly/art_daily_small_noise.csv': [],
 'artificialNoAnomaly/art_flatline.csv': [],
 'artificialNoAnomaly/art_noisy.csv': [],
 'artificialWithAnomaly/art_daily_flatmiddle.csv': [['2014-04-10 '
                                                     '07:15:00.000000',
                                                     '2014-04-11 '
                                                     '16:45:00.000000']],
 'artificialWithAnomaly/art_daily_jumpsdown.csv': [['2014-04-10 '
                                                    '16:15:00.000000',
                                                    '2014-04-12 '
                                                    '01:45:00.000000']],
 'artificialWithAnomaly/art_daily_jumpsup.csv': [['2014-04-10 16:15:00.000000',
                                                  '2014-04-12 '
                                              

In [6]:
def is_in_any_window(timestamp: pd.Timestamp, windows: list[list[str]]) -> bool:
    """
    Kiểm tra một timestamp có nằm trong một windows không
    """
    for start, end in windows:
        if pd.Timestamp(start) <= timestamp <= pd.Timestamp(end):
            return True

    return False

In [24]:
type(load_windows())

dict

In [7]:
type(WINDOWS_FILE)

str

In [14]:
def process_file(filepath: str, windows: list[list[str]]) -> pd.DataFrame:
    """
    Đọc một file csv và trả về một DataFrame được gán nhãn anomaly
    """
    df = pd.read_csv(filepath)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['is_anomaly'] = [is_in_any_window(t, windows) for t in df['timestamp']]
    return df

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

total_rows = 0
total_anomalies = 0

In [20]:
file_names = os.listdir(DATA_DIR)
file_names

['ec2_cpu_utilization_24ae8d.csv',
 'ec2_cpu_utilization_53ea38.csv',
 'ec2_cpu_utilization_5f5533.csv',
 'ec2_cpu_utilization_77c1ca.csv',
 'ec2_cpu_utilization_825cc2.csv',
 'ec2_cpu_utilization_ac20cd.csv',
 'ec2_cpu_utilization_c6585a.csv',
 'ec2_cpu_utilization_fe7f93.csv',
 'ec2_disk_write_bytes_1ef3de.csv',
 'ec2_disk_write_bytes_c0d644.csv',
 'ec2_network_in_257a54.csv',
 'ec2_network_in_5abac7.csv',
 'elb_request_count_8c0756.csv',
 'grok_asg_anomaly.csv',
 'iio_us-east-1_i-a2eb1cd9_NetworkIn.csv',
 'rds_cpu_utilization_cc0c53.csv',
 'rds_cpu_utilization_e47b3b.csv']

In [61]:
file_name: str
all_windows: dict = load_windows()

for file_name in file_names:
    
    if not file_name.endswith('.csv'):
        continue

    key = 'realAWSCloudwatch' + '/' + file_name
    windows = all_windows.get(key, [])  # tham số thứ 2 trả về giá trị mặc định (là []) nếu không tìm đươc key

    filepath: str = os.path.join(DATA_DIR, file_name)
    df = process_file(filepath, windows)

    print(df.head(5))

    values = df['is_anomaly'].unique()
    print(f'This csv file contains anomaly? {"Yes" if True in values else "No"} \n')

    # for row in df.itertuples():
    #     print(row)
    #     break

    rows: list[tuple[pd.Timestamp, str, float, bool]] = [
        (row.timestamp, file_name, row.value, bool(row.is_anomaly))
        for row in df.itertuples()
    ]

    for row in rows:
        print('First row:', row, '\n')
        break

    execute_values(
        cur,
        "INSERT INTO metrics_labeled (ts, source_file, value, is_anomaly) VALUES %s",
        rows,
    )

    conn.commit()

    n_anomaly = df['is_anomaly'].sum()
    total_rows += len(df)
    total_anomalies += n_anomaly

    print(f'{file_name}: {len(df)} rows, {n_anomaly} anomalies \n=========================================================\n') 


            timestamp  value  is_anomaly
0 2014-02-14 14:30:00  0.132       False
1 2014-02-14 14:35:00  0.134       False
2 2014-02-14 14:40:00  0.134       False
3 2014-02-14 14:45:00  0.134       False
4 2014-02-14 14:50:00  0.134       False
This csv file contains anomaly? Yes 

First row: (Timestamp('2014-02-14 14:30:00'), 'ec2_cpu_utilization_24ae8d.csv', 0.132, False) 

ec2_cpu_utilization_24ae8d.csv: 4032 rows, 402 anomalies 

            timestamp  value  is_anomaly
0 2014-02-14 14:30:00  1.732       False
1 2014-02-14 14:35:00  1.732       False
2 2014-02-14 14:40:00  1.960       False
3 2014-02-14 14:45:00  1.732       False
4 2014-02-14 14:50:00  1.706       False
This csv file contains anomaly? Yes 

First row: (Timestamp('2014-02-14 14:30:00'), 'ec2_cpu_utilization_53ea38.csv', 1.732, False) 

ec2_cpu_utilization_53ea38.csv: 4032 rows, 402 anomalies 

            timestamp   value  is_anomaly
0 2014-02-14 14:27:00  51.846       False
1 2014-02-14 14:32:00  44.508       Fa

In [62]:
cur.close()
conn.close()

# 2. Train model

In [63]:
import pandas as pd
import psycopg2
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

In [64]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "metrics_db",
    "user": "mlops",
    "password": "mlops123",
}

In [80]:
def load_data(source_file: str) -> pd.DataFrame:
    """
    Nhận một tên file (string) và truy vấn từ database lấy rows ứng với file đó
    """
    conn = psycopg2.connect(**DB_CONFIG)
    df = pd.read_sql(
        "SELECT ts, value, is_anomaly " \
        "FROM metrics_labeled " \
        "WHERE source_file= %(f)s " \
        "ORDER BY ts",
        conn,
        params={'f': source_file}
    )
    conn.close()
    return df

In [81]:
def build_features(df: pd.DataFrame, window=12) -> pd.DataFrame:
    """
    Sử dụng Rolling mean/std làm feature
    """
    # rolling: (với window=12)
    # giá trị dòng hiện tại được tính dựa trên dòng đó + 11 dòng trước đó
    # với mean: dòng đầu tiên có giá trị bằng chính dòng đó, dòng thứ 2 la fmean của 2 dòng đầu
    # với std: dòng đầu tiên ko tính được vì pandas cần tối thiểu 2 giá trị để tính standard deviation
    df['roll_mean'] = df['value'].rolling(window, min_periods=1).mean()
    df['roll_std']  = df['value'].rolling(window, min_periods=1).std().fillna(0)
    # hàm diff tính toán thay đổi của dòng này với dòng trước đó
    df['diff']      = df['value'].diff().fillna(0)
    return df        

In [130]:
def train_and_evaluation(source_file: str) -> tuple[IsolationForest, dict]:
    df = load_data(source_file=source_file)

    if (len(df) < 50):
        print('skip :D')
        return None

    df = build_features(df=df)
    feature_cols = ['value', 'roll_mean', 'roll_std', 'diff']

    X = df[feature_cols]
    y_true = df['is_anomaly'].astype(int)

    print(df.head(4))
    print(X.head(4))
    print(y_true.head(4))
    print('\n----------------------\n')

    # contamination là tỉ lệ dữ liệu bất thường
    # chẳng hạn contamination=0.001 thì IF lấy 0.1% data có anomaly score cao nhất
    # và gán là anomaly
    contamination = max(y_true.mean(), 0.001)
    isolation_forest_model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
    )

    isolation_forest_model.fit(X)

    # prediction
    # 1 là inliner, -1 là outliner/anomaly
    predictions: list[int] = isolation_forest_model.predict(X)
    print("yes there is anomaly predicted value" if -1 in predictions else "no")

    # y_pred đưa 1 (giá trị dự doán của mô hình ~ inliner) → 0 (ko phải anomaly)
    # y_pred đưa -1 (giá trị dự doán của mô hình ~ outliner) → 1 (là anomaly)
    y_pred = (predictions == -1).astype(int)
    print(y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"For source file: {source_file} - precision: {precision:.3f} recall: {recall:.3f} f1: {f1:.3f}")
    print(f'(n_anomaly={y_true.sum()}/{len(y_true)})')

    print("=======================================================\n\n\n\n")

    return isolation_forest_model, {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

### Train

Get source_files from database using query

In [83]:
conn = psycopg2.connect(**DB_CONFIG)
files: pd.DataFrame = pd.read_sql(
    "SELECT DISTINCT source_file " \
    "FROM metrics_labeled",
    conn,
)
files: list[str] = files['source_file'].tolist()
conn.close()
files

C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\3124983869.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  files: pd.DataFrame = pd.read_sql(


['ec2_cpu_utilization_24ae8d.csv',
 'ec2_cpu_utilization_53ea38.csv',
 'ec2_cpu_utilization_5f5533.csv',
 'ec2_cpu_utilization_77c1ca.csv',
 'ec2_cpu_utilization_825cc2.csv',
 'ec2_cpu_utilization_ac20cd.csv',
 'ec2_cpu_utilization_c6585a.csv',
 'ec2_cpu_utilization_fe7f93.csv',
 'ec2_disk_write_bytes_1ef3de.csv',
 'ec2_disk_write_bytes_c0d644.csv',
 'ec2_network_in_257a54.csv',
 'ec2_network_in_5abac7.csv',
 'elb_request_count_8c0756.csv',
 'grok_asg_anomaly.csv',
 'iio_us-east-1_i-a2eb1cd9_NetworkIn.csv',
 'rds_cpu_utilization_cc0c53.csv',
 'rds_cpu_utilization_e47b3b.csv']

In [132]:
results = []
best_model = None
best_f1 = -1
best_file = None

In [133]:
for file_name in files:
    result = train_and_evaluation(source_file=file_name)

    if result is None:
        continue

    isolation_forest_model, metrics = result

    results.append({
        'file': file_name,
        **metrics
    })

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_model = isolation_forest_model
        best_file = file_name

C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\4194878236.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


                         ts  value  is_anomaly  roll_mean  roll_std   diff
0 2014-02-14 14:30:00+00:00  0.132       False   0.132000  0.000000  0.000
1 2014-02-14 14:35:00+00:00  0.134       False   0.133000  0.001414  0.002
2 2014-02-14 14:40:00+00:00  0.134       False   0.133333  0.001155  0.000
3 2014-02-14 14:45:00+00:00  0.134       False   0.133500  0.001000  0.000
   value  roll_mean  roll_std   diff
0  0.132   0.132000  0.000000  0.000
1  0.134   0.133000  0.001414  0.002
2  0.134   0.133333  0.001155  0.000
3  0.134   0.133500  0.001000  0.000
0    0
1    0
2    0
3    0
Name: is_anomaly, dtype: int64

----------------------

yes there is anomaly predicted value
[0 0 0 ... 0 0 0]
For source file: ec2_cpu_utilization_24ae8d.csv - precision: 0.177 recall: 0.177 f1: 0.177
(n_anomaly=402/4032)




                         ts  value  is_anomaly  roll_mean  roll_std   diff
0 2014-02-14 14:30:00+00:00  1.732       False      1.732  0.000000  0.000
1 2014-02-14 14:35:00+00:00  1.732 

C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\4194878236.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


yes there is anomaly predicted value
[1 1 0 ... 0 0 0]
For source file: ec2_cpu_utilization_53ea38.csv - precision: 0.251 recall: 0.251 f1: 0.251
(n_anomaly=402/4032)




                         ts   value  is_anomaly  roll_mean  roll_std   diff
0 2014-02-14 14:27:00+00:00  51.846       False    51.8460  0.000000  0.000
1 2014-02-14 14:32:00+00:00  44.508       False    48.1770  5.188750 -7.338
2 2014-02-14 14:37:00+00:00  41.244       False    45.8660  5.429892 -3.264
3 2014-02-14 14:42:00+00:00  48.568       False    46.5415  4.634762  7.324
    value  roll_mean  roll_std   diff
0  51.846    51.8460  0.000000  0.000
1  44.508    48.1770  5.188750 -7.338
2  41.244    45.8660  5.429892 -3.264
3  48.568    46.5415  4.634762  7.324
0    0
1    0
2    0
3    0
Name: is_anomaly, dtype: int64

----------------------



C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\4194878236.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


...

yes there is anomaly predicted value
[1 1 1 ... 0 0 0]
For source file: iio_us-east-1_i-a2eb1cd9_NetworkIn.csv - precision: 0.381 recall: 0.381 f1: 0.381
(n_anomaly=126/1243)




                         ts  value  is_anomaly  roll_mean  roll_std   diff
0 2014-02-14 14:30:00+00:00  6.456       False      6.456  0.000000  0.000
1 2014-02-14 14:35:00+00:00  5.816       False      6.136  0.452548 -0.640
2 2014-02-14 14:40:00+00:00  6.268       False      6.180  0.328950  0.452
3 2014-02-14 14:45:00+00:00  5.816       False      6.089  0.324442 -0.452
   value  roll_mean  roll_std   diff
0  6.456      6.456  0.000000  0.000
1  5.816      6.136  0.452548 -0.640
2  6.268      6.180  0.328950  0.452
3  5.816      6.089  0.324442 -0.452
0    0
1    0
2    0
3    0
Name: is_anomaly, dtype: int64

----------------------



C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\4194878236.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


yes there is anomaly predicted value
[1 0 0 ... 1 1 1]
For source file: rds_cpu_utilization_cc0c53.csv - precision: 0.291 recall: 0.291 f1: 0.291
(n_anomaly=402/4032)




                         ts   value  is_anomaly  roll_mean  roll_std   diff
0 2014-04-10 00:02:00+00:00  14.012       False  14.012000  0.000000  0.000
1 2014-04-10 00:07:00+00:00  13.334       False  13.673000  0.479418 -0.678
2 2014-04-10 00:12:00+00:00  15.000       False  14.115333  0.837793  1.666
3 2014-04-10 00:17:00+00:00  13.998       False  14.086000  0.686566 -1.002
    value  roll_mean  roll_std   diff
0  14.012  14.012000  0.000000  0.000
1  13.334  13.673000  0.479418 -0.678
2  15.000  14.115333  0.837793  1.666
3  13.998  14.086000  0.686566 -1.002
0    0
1    0
2    0
3    0
Name: is_anomaly, dtype: int64

----------------------



C:\Users\Nguyen Quoc Duong\AppData\Local\Temp\ipykernel_26612\4194878236.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


yes there is anomaly predicted value
[1 0 1 ... 0 0 0]
For source file: rds_cpu_utilization_e47b3b.csv - precision: 0.169 recall: 0.169 f1: 0.169
(n_anomaly=402/4032)






In [134]:
results_df = pd.DataFrame(results)

print('SUMMARY: \n')
print(results_df.describe())

joblib.dump(best_model, "models/isolation_forest_nab_v1.pkl")
print(f"\nBest model (from {best_file}, f1={best_f1:.3f}) saved to models/isolation_forest_nab_v1.pkl")


SUMMARY: 

       precision     recall         f1
count  17.000000  17.000000  17.000000
mean    0.209408   0.209377   0.209392
std     0.094188   0.094188   0.094188
min     0.000000   0.000000   0.000000
25%     0.169154   0.169154   0.169154
50%     0.183623   0.183623   0.183623
75%     0.246269   0.246269   0.246269
max     0.416910   0.416910   0.416910

Best model (from ec2_cpu_utilization_825cc2.csv, f1=0.417) saved to models/isolation_forest_nab_v1.pkl


# 3. Problem??

- NAB dataset có hệ thống chấm điểm riêng

- có 17 model riêng biệt được train (mỗi model có thể tương ứng với một metrics khác nhau như disk write, cpu usage, etc.) → đang lưu lại một model duy nhất (mà có điểm f1 score tốt nhất)